# 01-Nowicki: 数据读入 + 质量控制

> 作者已完成 QC + normalization。本 notebook 仅做数据读入、格式对齐与基础标记，不做任何过滤。

In [ ]:
# === PARAMS ===
MANIFEST_PATH = "data/nowicki/manifest.yaml"
RUN_ID = "01-nowicki-v1-run001"  # 每次调参改用新 ID，禁止覆盖旧 run
RUN_ROOT = "results/runs"
OUTPUT_FILENAME = "01_nowicki_v1.h5ad"
QC_STRATEGY = "skip"
SCORE_CELL_CYCLE = True
RANDOM_SEED = 42
OUTPUT_VERSION = 1

In [ ]:
# === Setup：sys.path + 导入依赖 ===
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, sha256_file, snapshot_effective_parameters, validate_expression_contract,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

from scrna_integration.io import sync_gene_ids


In [ ]:
# 数据读入：h5ad 格式（Nowicki 2023，ensembl 基因，已 QC + normalized）
# 替代原来的 read_with_manifest / 按透明性铁律，数据读取逻辑拆回 cell
import yaml
from scrna_integration.io import sync_gene_ids

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
source_dataset = str(manifest["source_dataset"])

# ---- 1. 读取 h5ad ----
path = manifest["input"]["path"]
adata = sc.read_h5ad(path)
print(f"原始文件: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

# ---- 2. obs_mapping：列重命名（作者原始列名 → 框架统一列名）----
obs_map = manifest.get("obs_mapping", {})
for target_col, source_col in obs_map.items():
    if source_col in adata.obs.columns:
        adata.obs[target_col] = adata.obs[source_col]
    else:
        print(f"  WARNING: obs_mapping 源列 '{source_col}' 不存在，跳过")
# 删除已被映射的原始列（避免下游混淆）
mapped_cols = [c for c in obs_map.values() if c in adata.obs.columns]
if mapped_cols:
    adata.obs.drop(columns=mapped_cols, inplace=True)
print(f"obs_mapping 已应用 ({len(obs_map)} 个字段)")

# ---- 3. 数据标识字段 ----
adata.obs["source_dataset"] = source_dataset
adata.obs["project_id"] = manifest.get("project_id", "")
adata.obs["disease_system"] = manifest.get("disease_system", "")

# ---- 4. Layer 2 字段填充（缺失的填 NaN）----
layer2_fields = ["disease", "disease_ontology_term_id", "tissue",
                 "tissue_ontology_term_id", "assay", "sex", "development_stage"]
for field in layer2_fields:
    if field not in adata.obs.columns:
        print(f"  [Layer2] 列 '{field}' 缺失，填充 NaN")
        adata.obs[field] = np.nan
    else:
        col = adata.obs[field]
        n_null = col.isna().sum()
        n_empty = (col.astype(str).str.strip() == "").sum()
        if n_null + n_empty > 0:
            print(f"  [Layer2] 列 '{field}' 有 {n_null + n_empty}/{len(col)} 个缺失或空值")

# ---- 5. Layer 1 确定性校验 ----
layer1_required = ["source_dataset", "project_id", "disease_system"]
missing_l1 = [f for f in layer1_required if f not in adata.obs.columns]
if missing_l1:
    raise ValueError(f"Layer 1 必需字段缺失: {missing_l1}")

# ---- 6. 基线 QC 指标 ----
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)
adata.obs["n_genes"] = (adata.X > 0).sum(axis=1).A1 if sp.issparse(adata.X) else (adata.X > 0).sum(axis=1)
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).flatten()
mt_mask = adata.var.index.str.startswith("MT-")
if mt_mask.any():
    adata.obs["pct_counts_mt"] = (
        np.asarray(adata.X[:, mt_mask].sum(axis=1)).flatten()
        / adata.obs["total_counts"].values * 100
    )
ribo_mask = adata.var.index.str.startswith(("RPS", "RPL"))
if ribo_mask.any():
    adata.obs["pct_counts_ribo"] = (
        np.asarray(adata.X[:, ribo_mask].sum(axis=1)).flatten()
        / adata.obs["total_counts"].values * 100
    )

# ---- 7. 基因 ID 同步：Ensembl → Symbol ----
# Nowicki 数据 var.index 为 Ensembl ID（ENSG...），需转为 gene symbol
# 这是 02_merged inner join 的前提——所有数据集的 var.index 必须统一为 symbol
sync_gene_ids(adata, gene_id_format="ensembl")

# ---- 8. 从 .raw.X 提取 counts，写 layers["counts"] ----
# Nowicki: X 已是 logcounts，raw counts 在 .raw.X。
# 提取整数 counts、校验非负整数 + shape 对齐后写入 layers["counts"]（CSR float32）。
if adata.raw is None:
    raise ValueError("Nowicki 数据必须包含 .raw.X 原始计数，未找到 .raw 属性")

raw_adata = adata.raw.to_adata()

# 基因维度不一致检查：.raw.X 的 var 必须与当前 adata.var 完全对齐，不一致直接 raise
if list(raw_adata.var_names) != list(adata.var_names):
    raise ValueError(
        f"Nowicki .raw.X 基因维度与当前 adata 不一致："
        f"raw {raw_adata.n_vars} genes vs current {adata.n_vars} genes。"
        f"基因维度不一致，停止处理。请检查 manifest 使用的文件是否完整。"
    )

# 提取 counts 矩阵
counts_matrix = raw_adata.X.copy()

# 校验非负整数
counts_data = counts_matrix.data if sp.issparse(counts_matrix) else np.asarray(counts_matrix).ravel()
if counts_data.size == 0:
    raise ValueError("counts matrix is empty")
if (counts_data < 0).any():
    raise ValueError("counts contain negative values")
if not np.allclose(counts_data % 1, 0):
    raise ValueError("counts contain non-integer values")

# 写入 layers["counts"]（CSR float32）
adata.layers["counts"] = sp.csr_matrix(counts_matrix, dtype=np.float32)

# 写入 expression_contract
adata.uns["expression_contract"] = {
    "x_scale": "normalized_log1p",
    "counts_layer": "counts",
    "counts_source": ".raw.X",
    "counts_validated": True,
    "counts_integer_check": "full",
    "soupx_layer": None,
    "processing_history": ["01_extract_counts_from_raw"],
    "stage": "01",
}
print(f"layers['counts'] 已建立: {adata.layers['counts'].shape}，来自 .raw.X")
print(f"expression_contract: x_scale=normalized_log1p, counts_source=.raw.X, stage=01")

# ---- 9. 记录预处理状态（供下游 03_normalized 读取）----
adata.uns["preprocessing_done"] = manifest.get("preprocessing_done", [])
adata.uns["qc_overrides"] = manifest.get("qc_overrides", {})

print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


## 细胞周期评分

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
# scanpy 默认的 S/G2M 基因是 gene symbol 格式，但部分数据集使用 ensembl ID
# 作为基因名（如 Nowicki 原始数据），导致 scanpy 找不到匹配基因而抛出
# ValueError。用 try/except 优雅降级——跳过评分并标记为 unknown，不阻断管线。
if SCORE_CELL_CYCLE:
    s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    try:
        sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
        print("细胞周期评分完成")
        print(adata.obs["phase"].value_counts())
    except ValueError as e:
        # scanpy 找不到匹配的基因（如数据使用 ensembl ID 而默认基因列表是 symbol）
        # 此时无法做细胞周期评分，将对应列填充为 NaN/unknown，不阻断下游分析
        print(f"⚠️ 细胞周期评分跳过: {e}")
        print("  原因: scanpy 默认 S/G2M 基因列表为 gene symbol，数据可能使用其他 ID 体系")
        adata.obs['S_score'] = np.nan
        adata.obs['G2M_score'] = np.nan
        adata.obs['phase'] = 'unknown'
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 基因复杂度

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_nowicki_complexity.png", dpi=150, bbox_inches="tight")
plt.show()


## 原作者标注列确认

In [ ]:
# 确认原作者标注列已正确注入
annotation_cols = [c for c in adata.obs.columns if c.startswith("cell_type_original_")]
print("原作者标注列:")
for col in annotation_cols:
    vals = adata.obs[col].dropna().unique()
    print(f"  {col}: {len(vals)} 个唯一值: {sorted(vals)[:15]}...")


## QC 报告（跳过模式）

In [ ]:
# QC 报告
qc_report = {
    "strategy": "skip",
    "note": "作者已完成 basic_filter + doublet_removal + normalization；重新过滤会造成科学错误。",
    "cells_total": int(adata.n_obs),
    "cells_removed": 0,
    "pct_removed": 0.0,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
}
adata.uns["qc_report_v1"] = qc_report
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint
import scipy.sparse as sp
# 先计算所有可见门禁；只有到保存阶段才占用 RUN_ID。

# F2修复：记录作者预处理状态到 uns，供下游 03_normalized 判断是否跳过标准化
# nowicki manifest 声明 preprocessing_done 含 "normalization"，不做记录则 03 会二次归一
import yaml
_mani_path = MANIFEST_PATH
if not os.path.isabs(_mani_path):
    _mani_path = os.path.join(_root, _mani_path)
with open(_mani_path) as _mf:
    _manifest_data = yaml.safe_load(_mf)
_pp_done = _manifest_data.get("preprocessing_done", [])
if _pp_done:
    adata.uns["preprocessing_done"] = _pp_done
    print(f"preprocessing_done 已记录: {_pp_done}")

# F3修复：基因 ID 轴统一性断言——检查 per-dataset 内基因名无大小写混用
# 过大写/小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因 共 {_total} 个。"
        f"请统一基因名大小写（如全部 .str.upper()）后再进入 merge。"
    )
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

_source_values = sorted(map(str, adata.obs["source_dataset"].dropna().unique())) if "source_dataset" in adata.obs.columns else []
# expression_contract 验证（决策 1/2 管线）
validate_expression_contract(adata, expected_scale="normalized_log1p")
assert adata.uns["expression_contract"]["counts_source"] == ".raw.X", (
    f"counts_source 应为 '.raw.X'，实际为 {adata.uns['expression_contract']['counts_source']!r}"
)
print("expression_contract 验证通过: counts_source=.raw.X, x_scale=normalized_log1p")

hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "source_dataset_unique": len(_source_values) == 1 and not adata.obs["source_dataset"].isna().any(),
    "source_matches_manifest": _source_values == [str(source_dataset)],
    "counts_layer_exists": "counts" in adata.layers,
    "counts_layer_sparse": adata.layers.get("counts") is not None and sp.issparse(adata.layers["counts"]),
    "counts_layer_dtype": adata.layers.get("counts") is not None and adata.layers["counts"].dtype == np.float32,
    "counts_layer_shape_ok": adata.layers.get("counts") is not None and adata.layers["counts"].shape == adata.X.shape,
    "counts_source_is_raw": adata.uns.get("expression_contract", {}).get("counts_source") == ".raw.X",
}
stage_status = determine_stage_status({}, hard_postconditions, allow_no_required_methods=True)
effective_parameters = snapshot_effective_parameters(globals(), exclude=("RSCRIPT_BIN", "R_AVAILABLE"), path_root=Path(_root))
runtime_provenance = collect_runtime_provenance(_root, ("anndata", "scanpy", "numpy", "pandas", "scipy"))
manifest_sha256 = sha256_file(MANIFEST_PATH)
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": RUN_ID, "stage": "01_qcd", "stage_status": stage_status.value,
    "source_dataset": str(source_dataset),
    "inputs": [{"path": MANIFEST_PATH, "sha256": manifest_sha256}],
    "effective_parameters": effective_parameters, "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
}
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
adata.uns["stage"] = "01_qcd"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
OUTPUT_PATH = str(promote_run(run_paths))
print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
del adata; gc.collect()
print("内存已释放。")
